# 06 — Analyse d'erreurs, fidélité et recommandations

**Projet** : Résumé de comptes-rendus d'intervention — encodeur-décodeur appris, baseline extractive publiée

## Objectifs pédagogiques

1. Lire le verdict contractuel sur le split de **test**, une seule fois, avec sa marge et ses références mesurées sur les mêmes lignes.
1. Décomposer la fidélité par type de fait : la moyenne cache toujours le type qui manque.
1. Relire les erreurs une par une — fait oublié, valeur inventée, phrase qui bute sur sa borne — et distinguer les trois familles de correctifs.
1. Écrire le rapport du projet (tableaux et figures compris) et formuler des recommandations attachées à un chiffre.

## 0. Environnement et corpus

Le notebook ajuste lui-même les stratégies (baseline extractive et modèle de démonstration) : il n'a besoin d'aucun artefact de `make train` pour tourner, et il écrit ses sorties dans `outputs/notebooks/`.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

# --- Racine du projet ---------------------------------------------------------------------------
# Le notebook s'exécute depuis `notebooks/` : on remonte d'un cran pour pouvoir importer `src`.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hydra import compose, initialize_config_dir  # noqa: E402
from hydra.core.global_hydra import GlobalHydra  # noqa: E402

from src.schemas.config import validate_config  # noqa: E402
from src.utils.config_access import node  # noqa: E402
from src.utils.logging import setup_logging  # noqa: E402
from src.utils.paths import ProjectPaths  # noqa: E402

# --- Réglages d'affichage -----------------------------------------------------------------------
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 170)
# Le projet journalise au niveau INFO ; un notebook noierait ses tableaux sous ces lignes.
setup_logging(level="WARNING")

# --- Configuration : exactement celle de `python -m src.main` -----------------------------------
# Les notebooks travaillent sur un corpus réduit (200 comptes-rendus), un budget
# d'époques réduit et une architecture réduite : l'exécution reste de quelques secondes, et les
# valeurs absolues d'un notebook ne sont **pas** celles de la référence publiée par le README.
NB_DOCUMENTS = 200
NB_EPOCHS = 8

GlobalHydra.instance().clear()
with initialize_config_dir(config_dir=str(PROJECT_ROOT / "conf"), version_base=None):
    CONFIG = validate_config(
        compose(
            config_name="config",
            overrides=[
                "mode=train",
                f"data.n_samples={NB_DOCUMENTS}",
                f"train.epochs={NB_EPOCHS}",
                "model.params.layers=1",
                "model.params.units=96",
                "model.params.pretraining.epochs=1",
                "+model.params.vocab_size=800",
                "+model.params.min_frequency=2",
                "train.callbacks.progress_bar=false",
                "train.callbacks.logging_every=100",
                "train.early_stopping.enabled=false",
                "seed=42",
                "log_level=WARNING",
            ],
        )
    )

PATHS = ProjectPaths.from_root(PROJECT_ROOT)
# Bac à sable : les notebooks génèrent **leur** corpus (réduit) et écrivent **leurs** artefacts sous
# `outputs/notebooks/`. `data/raw`, `artifacts/` et donc les chiffres publiés par le README restent
# ceux de `make data` / `make train` : aucun notebook ne peut les modifier par accident.
NB_ROOT = PROJECT_ROOT / "outputs" / "notebooks"
NB_PATHS = ProjectPaths(
    root=PROJECT_ROOT,
    data_dir=NB_ROOT / "data",
    artifacts_dir=NB_ROOT / "artifacts",
).ensure()

MODEL_NODE = dict(node(CONFIG, "model"))
TEXT_COLUMN = str(MODEL_NODE.get("text_column", "text"))
ID_COLUMN = str(MODEL_NODE.get("id_column", "doc_id"))
COMPRESSION = float(MODEL_NODE.get("params", {}).get("compression", 0.45))
MAX_OUTPUT_TOKENS = int(MODEL_NODE.get("params", {}).get("max_output_tokens", 90))
STRATEGIES = list(MODEL_NODE.get("strategies_to_compare", []) or ["lead"])

print(f"Projet            : {CONFIG.project.name}")
print(f"Tâche             : {CONFIG.metrics.task}")
print(f"Métrique primaire : {CONFIG.metrics.primary} (seuil contractuel : 0.45)")
print(f"Algorithme servi  : {CONFIG.model.algorithm} ({CONFIG.model.name})")
print(f"Stratégies comparées : {', '.join(STRATEGIES)}")
print(f"Budget de longueur: compression {COMPRESSION:.2f}, plafond {MAX_OUTPUT_TOKENS} tokens")
print(f"Sorties notebook  : {NB_PATHS.artifacts_dir.relative_to(PROJECT_ROOT)}")

In [ ]:
from src.data.generators import SyntheticSummaryCorpusGenerator
from src.data.loaders import SummaryCorpusLoader

LOADER = SummaryCorpusLoader(
    NB_PATHS,
    dataset_name=str(CONFIG.data.dataset_name),
    formats=CONFIG.data.formats,
    validation_enabled=CONFIG.data.validation.raw,
    lazy_validation=CONFIG.data.validation.lazy,
)
if LOADER.documents_path.exists():
    DOCUMENTS, REFERENCES, FACTS = LOADER.load_corpus()
    METADATA = LOADER.load_metadata()
    print(
        f"Corpus du notebook lu : {len(DOCUMENTS)} comptes-rendus, "
        f"{len(REFERENCES)} résumés de référence, {len(FACTS)} faits annotés"
    )
else:
    # Un clone frais n'a pas encore de corpus réduit : le notebook le génère, de façon déterministe,
    # dans son bac à sable. `make data` produit le corpus de référence (600 comptes-rendus) dans
    # `data/raw` — les deux ne se mélangent jamais.
    bundle = SyntheticSummaryCorpusGenerator.from_config(CONFIG.data).generate()
    DOCUMENTS, REFERENCES, FACTS = bundle.documents, bundle.queries, bundle.facts
    METADATA = bundle.metadata
    LOADER.save_documents(DOCUMENTS)
    LOADER.save_references(REFERENCES)
    LOADER.save_facts(FACTS)
    LOADER.save_metadata(METADATA)
    print(f"Corpus réduit généré dans outputs/notebooks/data : {len(DOCUMENTS)} comptes-rendus")

TRAIN_DOCUMENTS, TRAIN_REFERENCES, TRAIN_FACTS = LOADER.load_split("train")
VAL_DOCUMENTS, VAL_REFERENCES, VAL_FACTS = LOADER.load_split("val")
TEST_DOCUMENTS, TEST_REFERENCES, TEST_FACTS = LOADER.load_split("test")
SPLIT_SIZES = LOADER.split_sizes()
# Le test ne sert **jamais** à régler quoi que ce soit : les réglages se lisent sur la validation,
# le verdict sur le test, une seule fois.
GOLDEN = dict(zip(REFERENCES["doc_id"], REFERENCES["summary"], strict=False))
print("splits :", SPLIT_SIZES)
print("compression moyenne demandée :", round(float(REFERENCES["compression"].mean()), 4))

## 1. Mesurer le test, une fois

L'évaluateur reçoit le modèle servi **et** ses références. Toutes les stratégies sont mesurées sur les mêmes lignes de test, avec la même référence : une comparaison entre deux exécutions différentes ne voudrait rien dire.

In [ ]:
from src.evaluation.evaluator import SummaryEvaluator
from src.models import build_model

SERVED = build_model(CONFIG, algorithm="transformer_tiny")
SERVED.fit(TRAIN_DOCUMENTS, TRAIN_REFERENCES, validation=(VAL_DOCUMENTS, VAL_REFERENCES))
baseline = build_model(CONFIG, algorithm="lead")
baseline.fit(TRAIN_DOCUMENTS, TRAIN_REFERENCES)

evaluator = SummaryEvaluator(
    SERVED,
    config=CONFIG.model_dump(),
    metrics_config=CONFIG.metrics.model_dump(),
    paths=NB_PATHS,
    text_column=TEXT_COLUMN,
    id_column=ID_COLUMN,
    baselines={"baseline_extractive": baseline},
)
EVALUATION = evaluator.evaluate(TEST_DOCUMENTS, TEST_REFERENCES, TEST_FACTS)
print("verdict :", EVALUATION.verdict)
print("détail  :", EVALUATION.verdict_detail)
display(EVALUATION.per_strategy.round(4))

**Ce qu'il faut retenir**

- Le verdict est lu sur la métrique principale déclarée par la configuration, avec son seuil : un rapport qui annonce un score sans dire à quoi il se compare n'est pas un verdict.
- La marge (observé moins seuil) est publiable telle quelle : elle dit si le système est juste au-dessus ou largement au-dessus, ce qui n'est pas la même information pour décider d'un déploiement.
- Les références sont mesurées **sur les mêmes lignes** : comparer le modèle au score publié dans un autre contexte serait une comparaison invalide.

## 2. La fidélité, type de fait par type de fait

La couverture résume en un chiffre ce que le résumé rapporte des faits saillants. Un type oublié reste invisible dans la moyenne : il faut donc la ventilation, et les valeurs non supportées — celles que le résumé **invente**.

In [ ]:
from src.data.schemas import fact_columns

fidelity = EVALUATION.fidelity
coverage_columns = [column for column in fact_columns() if column in fidelity.columns]
by_type = (
    fidelity[coverage_columns]
    .mean()
    .rename(lambda name: name.replace("covered_", ""))
    .sort_values()
)
display(by_type.round(4).to_frame("couverture moyenne"))

print("couverture globale   :", round(float(fidelity["fact_coverage"].mean()), 4))
print("précision des faits  :", round(float(fidelity["fact_precision"].mean()), 4))
print("valeurs non supportées (total) :", int(fidelity["n_unsupported"].sum()))
print(
    "résumés contenant au moins une valeur inventée :",
    int((fidelity["n_unsupported"] > 0).sum()),
    "/",
    len(fidelity),
)

# La couverture se lit **contre** celle des stratégies extractives, mesurée sur les mêmes lignes :
# une extractive ne peut pas inventer, donc son score est le plancher honnête de cette colonne.
extractives = EVALUATION.per_strategy.loc[EVALUATION.per_strategy["strategy"] != SERVED.strategy]
if not extractives.empty and "fact_coverage" in extractives.columns:
    best = extractives.sort_values("fact_coverage", ascending=False).iloc[0]
    print(
        "meilleure couverture extractive :",
        round(float(best["fact_coverage"]), 4),
        f"({best['strategy']}) | modèle servi :",
        round(float(fidelity["fact_coverage"].mean()), 4),
    )

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
by_type.plot.bar(ax=axes[0], rot=30, title="Couverture par type de fait (test)")
fidelity["fact_coverage"].plot.hist(ax=axes[1], bins=20, title="Couverture par document")
plt.tight_layout()
plt.show()

**Ce qu'il faut retenir**

- Un type de fait peu couvert est un correctif ciblé : soit il est rare dans le train (peu d'exemples à apprendre), soit il n'est pas reformulé de la même façon que dans la référence.
- La précision des faits et la couverture se lisent ensemble : couvrir beaucoup en inventant n'est pas une amélioration, c'est un déplacement du problème.
- L'annotation ne couvre que les valeurs **vérifiables** (équipement, durée, statut…) : la détection d'hallucination s'arrête là où l'annotation s'arrête, et le rapport le dit.

## 3. Où le modèle échoue

Les segments sont les colonnes du corpus (type d'intervention, urgence, site) : ils répondent à la question « pour qui le modèle est-il moins bon ? », qui précède « pourquoi ? ».

In [ ]:
segments = EVALUATION.segments
for column in sorted(segments["segment"].unique()) if not segments.empty else []:
    pivot = segments[segments["segment"] == column].sort_values("rouge1_f")
    display(pivot[["value", "n_documents", "rouge1_f", "fact_coverage", "compression"]].round(4))

In [ ]:
if not segments.empty:
    fig, axes = plt.subplots(
        1, len(sorted(segments["segment"].unique())), figsize=(14, 4), squeeze=False
    )
    for axis, column in zip(axes[0], sorted(segments["segment"].unique()), strict=False):
        subset = segments[segments["segment"] == column].sort_values("rouge1_f")
        axis.bar(subset["value"], subset["rouge1_f"])
        axis.set_title(f"ROUGE-1 par {column}")
        axis.tick_params(axis="x", rotation=30)
    plt.tight_layout()
    plt.show()
else:
    print("aucun segment déclaré par la configuration")

## 4. Les erreurs, relues une par une

Le rapport archive les documents où le modèle est le plus loin de sa référence, avec les faits saillants qu'il aurait fallu rapporter. C'est là qu'on distingue un **fait oublié** (couverture basse, ROUGE correct) d'une **valeur inventée** (précision basse) et d'une **phrase qui bute sur sa borne**.

In [ ]:
errors = EVALUATION.errors
print("documents archivés :", len(errors))
display(
    errors.head(5)[
        [
            c
            for c in errors.columns
            if c in ("doc_id", "rouge1_f", "fact_coverage", "unsupported_facts", "hit_max_length")
        ]
    ]
)

worst = errors.iloc[0]
worst_id = str(worst["doc_id"])
worst_reference = TEST_REFERENCES.loc[TEST_REFERENCES["doc_id"] == worst_id, "summary"].iloc[0]
worst_fidelity = fidelity.loc[fidelity["doc_id"] == worst_id]

print(
    f"document {worst_id} : ROUGE-1 {float(worst['rouge1_f']):.4f}, "
    f"couverture {float(worst['fact_coverage']):.4f}"
)
print("faits saillants du document :")
display(
    TEST_FACTS[(TEST_FACTS["doc_id"] == worst_id) & (TEST_FACTS["salient"] == 1)][
        ["fact_type", "value", "sentence_index"]
    ]
)
print("résumé de référence :", worst_reference[:400])
print(
    "faits rapportés par le modèle :",
    int(worst_fidelity["n_covered"].sum()),
    "/",
    int(worst_fidelity["n_salient"].sum()),
)
if "unsupported_values" in worst_fidelity.columns:
    print("valeurs inventées :", str(worst_fidelity["unsupported_values"].iloc[0]) or "aucune")

In [ ]:
summary_of_worst = EVALUATION.predictions.loc[
    (EVALUATION.predictions["doc_id"] == worst_id)
    & (EVALUATION.predictions["strategy"] == SERVED.strategy)
]["prediction"]
print(
    "résumé produit par le modèle :",
    "" if summary_of_worst.empty else summary_of_worst.iloc[0][:400],
)
print(
    "\npart des résumés qui atteignent leur budget de longueur (test) :",
    float(
        EVALUATION.predictions.loc[
            EVALUATION.predictions["strategy"] == SERVED.strategy, "hit_max_length"
        ].mean()
    ),
)
print(
    "compression moyenne du modèle  :",
    round(
        float(
            EVALUATION.predictions.loc[
                EVALUATION.predictions["strategy"] == SERVED.strategy, "compression"
            ].mean()
        ),
        4,
    ),
)
print(
    "longueur moyenne de la référence :",
    round(float(TEST_REFERENCES["compression"].mean()), 4),
    "(compression)",
)

**Ce qu'il faut retenir**

- Trois familles d'erreurs, trois correctifs : un fait **oublié** se corrige par des données ou du budget ; une valeur **inventée** par une contrainte de fidélité ; une phrase **tronquée** par le budget ou par le critère d'arrêt du décodeur.
- Un ROUGE bas avec une couverture correcte signale une reformulation différente de la référence, pas une contre-vérité : le corpus accepte plusieurs formulations, la référence n'en connaît qu'une.
- Relire l'erreur « la pire » est ce qui empêche de conclure sur une moyenne : un exemple suffit souvent à identifier la cause, et il se cite.

## 5. Le rapport du projet

Le constructeur de rapport transforme l'évaluation en artefacts : un rapport Markdown lisible, des tableaux CSV pour un portail de CI et les figures. Ce sont les mêmes fichiers que `make evaluate` écrit.

In [ ]:
from src.evaluation.reports import ReportBuilder

bundle = ReportBuilder(CONFIG.model_dump(), paths=NB_PATHS).build(EVALUATION)
print("rapport  :", bundle.report.relative_to(PROJECT_ROOT))
for name, path in sorted(bundle.tables.items()):
    print(f"  tableau  : {name:20s} {path.relative_to(PROJECT_ROOT)}")
for name, path in sorted(bundle.figures.items()):
    print(f"  figure   : {name:20s} {path.relative_to(PROJECT_ROOT)}")

print("\n" + "=" * 90)
print(bundle.report.read_text(encoding="utf-8")[:1500])

## 6. Recommandations

Chaque recommandation est attachée au chiffre qui la justifie. Une recommandation sans chiffre est une opinion, et ce notebook n'en publie pas.

In [ ]:
observed = float(EVALUATION.verdict_detail.get("observed", 0.0))
threshold = EVALUATION.verdict_detail.get("threshold")
weakest = by_type.index[0]
recommendations = [
    f"1. Le seuil contractuel ({threshold}) n'est pas atteint sur ce run de démonstration "
    f"({observed:.4f}) : `mode=all` entraîne le modèle de référence et publie le verdict réel.",
    f"2. Le type de fait le plus faible est « {weakest} » ({by_type.iloc[0]:.4f} de couverture) : "
    "c'est là qu'un ajustement du corpus ou du budget de longueur a le plus de rendement.",
]
if int(fidelity["n_unsupported"].sum()) > 0:
    recommendations.append(
        f"3. {int(fidelity['n_unsupported'].sum())} valeur(s) inventée(s) détectée(s) : renforcer "
        "la contrainte de fidélité avant d'augmenter la capacité du modèle."
    )
else:
    recommendations.append(
        "3. Aucune valeur inventée détectée sur ce run : la fidélité mesurable est tenue, la "
        "marge de progression est dans la couverture."
    )
served_predictions = EVALUATION.predictions.loc[
    EVALUATION.predictions["strategy"] == SERVED.strategy
]
budget_share = float(served_predictions["hit_max_length"].mean())
recommendations.append(
    f"4. Budget de longueur : {budget_share:.1%} des résumés atteignent leur borne, contre une "
    f"compression de référence de {float(TEST_REFERENCES['compression'].mean()):.4f} : vérifier "
    "le critère d'arrêt du décodeur avant d'accuser le modèle."
)
if not extractives.empty and "fact_coverage" in extractives.columns:
    recommendations.append(
        f"5. Couverture des faits : {float(fidelity['fact_coverage'].mean()):.4f} pour le modèle "
        f"servi contre {float(extractives['fact_coverage'].max()):.4f} pour la meilleure "
        "extractive, sur les mêmes lignes — un extracteur recopie les valeurs par construction, le "
        "démonstrateur réduit doit encore apprendre à le faire."
    )
for line in recommendations:
    print(line)

**Ce qu'il faut retenir**

- Un verdict « non conforme » n'est pas un échec du projet : c'est le résultat d'une mesure, publié avec sa marge. Le run de référence (`mode=all`), lui, entraîne le modèle complet et affiche son propre verdict.
- Les recommandations sont ordonnées par rendement : d'abord le type de fait le plus faible, ensuite la fidélité, enfin le budget — jamais « améliorer le modèle » sans chiffre.
- Tous les artefacts écrits ici sont dans `outputs/notebooks/` : le rapport publié par le projet reste celui de `make evaluate`, et les deux ne se mélangent pas.